<a href="https://colab.research.google.com/github/ryosuke-yakura/kaggle_jigsaw/blob/main/250906_DeBert_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
#. ライブラリのインポート
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import sklearn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
import datasets
import transformers
from transformers import pipeline
from transformers import DataCollatorWithPadding
from transformers import get_linear_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel, TrainingArguments, Trainer

In [2]:
#. data_read
train = pd.read_csv("/content/drive/MyDrive/Kaggle_Jigsaw/data/train.csv")
test = pd.read_csv("/content/drive/MyDrive/Kaggle_Jigsaw/data/test.csv")
sample_submission = pd.read_csv("/content/drive/MyDrive/Kaggle_Jigsaw/data/sample_submission.csv")

In [3]:
#. data_check
def data_check(df):
  print(df.shape)
  display(df.head())
  display(df.isnull().sum())

In [4]:
data_check(train)
data_check(test)
data_check(sample_submission)

(2029, 9)


,row_id,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2,rule_violation
0,0,Banks don't want you to know this! Click here ...,"No Advertising: Spam, referral links, unsolici...",Futurology,If you could tell your younger self something ...,hunt for lady for jack off in neighbourhood ht...,Watch Golden Globe Awards 2017 Live Online in ...,"DOUBLE CEE x BANDS EPPS - ""BIRDS""\n\nDOWNLOAD/...",0
1,1,SD Stream [ ENG Link 1] (http://www.sportsstre...,"No Advertising: Spam, referral links, unsolici...",soccerstreams,[I wanna kiss you all over! Stunning!](http://...,LOLGA.COM is One of the First Professional Onl...,#Rapper \n🚨Straight Outta Cross Keys SC 🚨YouTu...,[15 Amazing Hidden Features Of Google Search Y...,0
2,2,Lol. Try appealing the ban and say you won't d...,No legal advice: Do not offer or request legal...,pcmasterrace,Don't break up with him or call the cops. If ...,It'll be dismissed: https://en.wikipedia.org/w...,Where is there a site that still works where y...,Because this statement of his is true. It isn'...,1
3,3,she will come your home open her legs with an...,"No Advertising: Spam, referral links, unsolici...",sex,Selling Tyrande codes for 3€ to paypal. PM. \n...,tight pussy watch for your cock get her at thi...,NSFW(obviously) http://spankbang.com/iy3u/vide...,Good News ::Download WhatsApp 2.16.230 APK for...,1
4,4,code free tyrande --->>> [Imgur](http://i.imgu...,"No Advertising: Spam, referral links, unsolici...",hearthstone,wow!! amazing reminds me of the old days.Well...,seek for lady for sex in around http://p77.pl/...,must be watch movie https://sites.google.com/s...,We're streaming Pokemon Veitnamese Crystal RIG...,1


,0
row_id,0
body,0
rule,0
subreddit,0
positive_example_1,0
positive_example_2,0
negative_example_1,0
negative_example_2,0
rule_violation,0


(10, 8)


,row_id,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2
0,2029,NEW RAP GROUP 17. CHECK US OUT https://soundcl...,"No Advertising: Spam, referral links, unsolici...",hiphopheads,"Hey, guys, just wanted to drop in and invite y...",Cum Swallowing Hottie Katrina Kaif Cartoon Xvi...,SD Stream Eng - [Chelsea TV USA](http://soccer...,HD Streams: |[ENG HD Stoke vs Manchester Unite...
1,2030,Make your life comfortable. Get up to 15% Disc...,No legal advice: Do not offer or request legal...,AskReddit,Get a lawyer and get the security camera foota...,That isn't drastic. You tried reaching out to ...,So what are you going to do with the insurance...,It's just for Austria & Germany. If you still ...
2,2031,Kickin' ass and selling underwear!\nJust made ...,"No Advertising: Spam, referral links, unsolici...",gonewild,Good story my friend. Check out my blog at ht...,If you know what exactly you need then you don...,CENTIPEDES\n\nSOME BASED PATRIOTS HAVE CREATED...,[So great! Thanks for sharing.](http://www.che...
3,2032,watch hooters best therein http://clickan...,"No Advertising: Spam, referral links, unsolici...",personalfinance,"Earn 50,000 bonus points with Chase Sapphire P...","Cool, front page! I made this print along with...",[Full HD Movie Online Free](http://www.flickma...,* Karambit Black Pearl\n* 0.02137822 Float (un...
4,2033,bitches for free at this point show all h...,"No Advertising: Spam, referral links, unsolici...",Showerthoughts,code free tyrande --->>> [Imgur](http://i.imgu...,My trade link\nhttps://steamcommunity.com/trad...,**HD** [ mio Stadium 102 HD](http://www.genti....,Infographics is an incredible method for showi...


,0
row_id,0
body,0
rule,0
subreddit,0
positive_example_1,0
positive_example_2,0
negative_example_1,0
negative_example_2,0


(10, 2)


,row_id,rule_violation
0,2029,0.5
1,2030,0.5
2,2031,0.5
3,2032,0.5
4,2033,0.5


,0
row_id,0
rule_violation,0


In [5]:
#. config
TRAINING_MODEL_PATH = 'microsoft/deberta-base-mnli' #. modelpath
TRAINING_MAX_LENGTH = 524 #. トークナイザする文字数
EVAL_MAX_LENGTH = 524 #. 最大の文字数
CONF_THRESH = 0.9
NUM_EPOCHS = 3
BATCH_SIZE = 4
EVAL_BATCH_SIZE = 2
OUTPUT_DIR = "output"
learning_rate = 5e-5
warmup_step_ratio = 0.01
num_workers = 0

In [6]:
#. モデルのロード
tokenizer = AutoTokenizer.from_pretrained(TRAINING_MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(TRAINING_MODEL_PATH).to("cuda")

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/557M [00:00<?, ?B/s]

Some weights of the model checkpoint at microsoft/deberta-base-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [7]:
#. 各カラムから抜き取る長さを調整
def trancate(sentence, max_tokens):
  if not sentence:
    return ""
  else:
    if len(sentence) <= max_tokens:
      return sentence
    else:
      return sentence[:max_tokens]

In [8]:
#. トークンを追加
SECTION_TOKENS = [
    "[RULE]", "[SUBREDDIT]", "[POS_EX1]", "[POS_EX2]", "[NEG_EX1]", "[NEG_EX2]", "[TEXT]"
]

# PAD が無い系（RoBERTa系など）の保険
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})

# 追加特殊トークンを登録
num_added = tokenizer.add_special_tokens({"additional_special_tokens": SECTION_TOKENS})
print("added tokens:", num_added)  # 0 の場合は既に登録済み

added tokens: 7


In [9]:
# トークナイザ側で語彙が増えたら必ず実行
model.resize_token_embeddings(len(tokenizer))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(50272, 768, padding_idx=0)

In [10]:
def make_input_row(body, rule, subreddit, pos1, pos2, neg1, neg2):
    # None/NaN を空文字にする簡易クリーナ
    def _c(x):
        return "" if x is None or str(x).lower() == "nan" else str(x).strip()

    parts = [
        f"[RULE] {_c(rule)}",
        f"[SUBREDDIT] {_c(subreddit)}",
        f"[POS_EX1] {_c(pos1)}",
        f"[POS_EX2] {_c(pos2)}",
        f"[NEG_EX1] {_c(neg1)}",
        f"[NEG_EX2] {_c(neg2)}",
        f"[TEXT] {_c(body)}",
    ]
    # 改行で区切ると可読性・デバッグ性が高い
    return "\n".join(parts)

In [11]:
def make_input_text_data(df):
  df["feature_text"] = [
      make_input_row(b, r, s, p1, p2, n1, n2)
      for b, r, s, p1, p2, n1, n2 in zip(
          df["body"], df["rule"], df["subreddit"],
          df["positive_example_1"], df["positive_example_2"],
          df["negative_example_1"], df["negative_example_2"]
      )
  ]
  return df

train_text = make_input_text_data(train)
train_text.head()

,row_id,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2,rule_violation,feature_text
0,0,Banks don't want you to know this! Click here ...,"No Advertising: Spam, referral links, unsolici...",Futurology,If you could tell your younger self something ...,hunt for lady for jack off in neighbourhood ht...,Watch Golden Globe Awards 2017 Live Online in ...,"DOUBLE CEE x BANDS EPPS - ""BIRDS""\n\nDOWNLOAD/...",0,"[RULE] No Advertising: Spam, referral links, u..."
1,1,SD Stream [ ENG Link 1] (http://www.sportsstre...,"No Advertising: Spam, referral links, unsolici...",soccerstreams,[I wanna kiss you all over! Stunning!](http://...,LOLGA.COM is One of the First Professional Onl...,#Rapper \n🚨Straight Outta Cross Keys SC 🚨YouTu...,[15 Amazing Hidden Features Of Google Search Y...,0,"[RULE] No Advertising: Spam, referral links, u..."
2,2,Lol. Try appealing the ban and say you won't d...,No legal advice: Do not offer or request legal...,pcmasterrace,Don't break up with him or call the cops. If ...,It'll be dismissed: https://en.wikipedia.org/w...,Where is there a site that still works where y...,Because this statement of his is true. It isn'...,1,[RULE] No legal advice: Do not offer or reques...
3,3,she will come your home open her legs with an...,"No Advertising: Spam, referral links, unsolici...",sex,Selling Tyrande codes for 3€ to paypal. PM. \n...,tight pussy watch for your cock get her at thi...,NSFW(obviously) http://spankbang.com/iy3u/vide...,Good News ::Download WhatsApp 2.16.230 APK for...,1,"[RULE] No Advertising: Spam, referral links, u..."
4,4,code free tyrande --->>> [Imgur](http://i.imgu...,"No Advertising: Spam, referral links, unsolici...",hearthstone,wow!! amazing reminds me of the old days.Well...,seek for lady for sex in around http://p77.pl/...,must be watch movie https://sites.google.com/s...,We're streaming Pokemon Veitnamese Crystal RIG...,1,"[RULE] No Advertising: Spam, referral links, u..."


In [12]:
#. max_length確認
_max = 0
for i in range(0, len(train_text)):
  if _max < len(train_text["feature_text"][i]):
    _max = len(train_text["feature_text"][i])

_max

2003

In [13]:
#. 学習、テストの切り分け
X = train_text["feature_text"].tolist()
y = train_text["rule_violation"].tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

In [14]:
#. トークン化
def proprocess_tokenizer(examples):
  return tokenizer(
      examples,
      truncation=True,
      padding=True,
      max_length=TRAINING_MAX_LENGTH
  )

train_encoding = proprocess_tokenizer(X_train)
val_encoding = proprocess_tokenizer(X_test)

In [21]:
class RedditDataset(torch.utils.data.Dataset):
  def __init__(self, encodings, labels):
    self.encodings = encodings
    self.labels = labels

  def __getitem__(self, idx):
    item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
    item["labels"] = torch.tensor(self.labels[idx])
    return item

  def __len__(self):
    return len(self.labels)

train_dataset = RedditDataset(train_encoding, y_train)
val_dataset = RedditDataset(val_encoding, y_test)

In [22]:
model

DebertaForSequenceClassification(
  (deberta): DebertaModel(
    (embeddings): DebertaEmbeddings(
      (word_embeddings): Embedding(50272, 768, padding_idx=0)
      (LayerNorm): DebertaLayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaLayer(
          (attention): DebertaAttention(
            (self): DisentangledSelfAttention(
              (in_proj): Linear(in_features=768, out_features=2304, bias=False)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (pos_proj): Linear(in_features=768, out_features=768, bias=False)
              (pos_q_proj): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): DebertaLayerNorm()
              (dropout): Dropout(p=0

In [23]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    warmup_steps=500,
    eval_strategy="epoch",
    logging_strategy="steps",
    logging_dir="./logs",
    report_to=[],
    disable_tqdm=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

In [26]:
#. 学習
trainer.train()

{'eval_loss': 1.0666934251785278, 'eval_runtime': 2.9235, 'eval_samples_per_second': 69.438, 'eval_steps_per_second': 34.89, 'epoch': 1.0}
{'loss': 0.4435, 'grad_norm': 38.29841613769531, 'learning_rate': 4.99e-05, 'epoch': 1.0940919037199124}
{'eval_loss': 1.1207813024520874, 'eval_runtime': 2.956, 'eval_samples_per_second': 68.674, 'eval_steps_per_second': 34.506, 'epoch': 2.0}
{'loss': 0.4261, 'grad_norm': 9.269731521606445, 'learning_rate': 2.135476463834673e-05, 'epoch': 2.1881838074398248}
{'eval_loss': 1.0306261777877808, 'eval_runtime': 2.9682, 'eval_samples_per_second': 68.392, 'eval_steps_per_second': 34.364, 'epoch': 3.0}
{'train_runtime': 247.5162, 'train_samples_per_second': 22.132, 'train_steps_per_second': 5.539, 'train_loss': 0.3884504500485788, 'epoch': 3.0}


TrainOutput(global_step=1371, training_loss=0.3884504500485788, metrics={'train_runtime': 247.5162, 'train_samples_per_second': 22.132, 'train_steps_per_second': 5.539, 'train_loss': 0.3884504500485788, 'epoch': 3.0})

In [29]:
#. 結果の確認
preds = trainer.predict(val_dataset)
probs = torch.nn.functional.softmax(torch.tensor(preds.predictions), dim=1)[:, 1].numpy()

auc= roc_auc_score(y_test, probs)
print(f"validation AUC: {auc:.4f}")

validation AUC: 0.8666


In [30]:
#. テストデータの前処理
test_text = make_input_text_data(test)
test_data = test_text["feature_text"].tolist()
test_encoding = proprocess_tokenizer(test_data)
test_dataset = RedditDataset(test_encoding, [0]*len(test_data))

In [31]:
#. 推論
preds = trainer.predict(test_dataset)
probs = torch.nn.functional.softmax(torch.tensor(preds.predictions), dim=1)[:, 1].numpy()

In [34]:
#. サブミットファイル作成
submission = pd.DataFrame({"row_id": test["row_id"], "rule_violation": probs})
submission.to_csv("submission.csv", index=False)

In [35]:
submission

,row_id,rule_violation
0,2029,0.002614
1,2030,0.437078
2,2031,0.997731
3,2032,0.997984
4,2033,0.997408
5,2034,0.001961
6,2035,0.997657
7,2036,0.001929
8,2037,0.005539
9,2038,0.997835
